# ULTRON v0.3 — training
Run the cells from top to bottom. This trains the custom ULTRON Transformer from scratch on the project dataset.

In [ ]:
!rm -rf /content/Ultron-ai\n!git clone -b ultron-brain-v0.1 https://github.com/yosimaracabal2013-stack/Ultron-ai.git /content/Ultron-ai\n%cd /content/Ultron-ai\n!pip -q install torch


In [ ]:
import os, sys\nsys.path.insert(0, '/content/Ultron-ai/ai')\nos.environ['ULTRON_GPU_STEPS']='5000'\nos.environ['ULTRON_CPU_STEPS']='500'\n!python ai/train_v0_3.py


In [ ]:
# Quantize the trained checkpoint to INT8 for compact deployment.\nimport torch, json\nfrom pathlib import Path\nckpt=Path('/content/Ultron-ai/ai/checkpoints/ultron_v0_3.pt')\nout=Path('/content/Ultron-ai/ai/checkpoints/ultron_v0_2.pt')\ndata=torch.load(ckpt,map_location='cpu',weights_only=False)\nq={}; scales={}\nfor name,t in data['model'].items():\n    if torch.is_floating_point(t):\n        scale=float(t.abs().max().item()/127.0) or 1e-8\n        q[name]=torch.clamp(torch.round(t/scale),-127,127).to(torch.int8)\n        scales[name]=scale\n    else:\n        q[name]=t\npacked={'format':'ultron-int8-v1','model_int8':q,'scales':scales,'config':data['config']}\ntorch.save(packed,out)\nprint('INT8 checkpoint:',out)\nprint('size MB:',out.stat().st_size/1024/1024)


In [ ]:
from google.colab import files\nfiles.download('/content/Ultron-ai/ai/checkpoints/ultron_v0_2.pt')\nfiles.download('/content/Ultron-ai/ai/checkpoints/tokenizer_v0_3.json')


After training, the two downloaded files are the new ULTRON brain files: `ultron_v0_3_int8.pt` and `tokenizer_v0_3.json`.

In [ ]:
# Quick local generation test before deployment\nimport torch, sys\nsys.path.insert(0,'/content/Ultron-ai/ai')\nfrom tokenizer import CharTokenizer\nfrom model import UltronTransformer\ntok=CharTokenizer.load('/content/Ultron-ai/ai/checkpoints/tokenizer_v0_3.json')\nraw=torch.load('/content/Ultron-ai/ai/checkpoints/ultron_v0_3.pt',map_location='cpu',weights_only=False)\nmodel=UltronTransformer(**raw['config']); model.load_state_dict(raw['model']); model.eval()\np='User: What is your purpose?\\nULTRON:'\nids=torch.tensor([[tok.stoi.get(c,tok.stoi.get(' ')) for c in p]],dtype=torch.long)\nwith torch.no_grad(): out=model.generate(ids,80,temperature=0.35,top_k=10)[0].tolist()\nprint(tok.decode(out[len(ids[0]):]))
